# Archiver dans HDFS puis charger le Data Warehouse

Ordre du sujet : `container_events.csv` est **archivé dans HDFS avant** son chargement.

**Avant de lancer :** Docker Desktop ouvert et cluster démarré (voir `README_HDFS_Docker.md`). Sans cluster, le notebook s'arrête à la première commande HDFS.

⚠️ La logique a été testée avec des commandes HDFS **simulées**. L'exécution sur le vrai cluster reste à faire : c'est pourquoi ce notebook est livré sans résultats.

Version notebook de `ingestion_depuis_hdfs.py`.

In [ ]:
from pathlib import Path

# Un notebook n'a pas de __file__ : on remonte les dossiers jusqu'à la racine du projet
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "03_data" / "raw").is_dir())
print("Projet :", ROOT.name)

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
from datetime import date, datetime
from pathlib import Path

HERE = ROOT / "04_scripts" / "hdfs"
RAW_DIR = ROOT / "03_data" / "raw"
EVENTS = "container_events.csv"
OTHERS = ["customers.csv", "vessel_calls.csv", "yard_moves.csv", "equipment.csv"]
HDFS_BASE = "/port_exam/project_01/input"


def hdfs(*args, check=True):
    """Exécute une commande dans le conteneur NameNode et renvoie le résultat (octets bruts)."""
    cmd = ["docker", "compose", "exec", "-T", "namenode", "hdfs", *args]
    print("$", " ".join(["hdfs", *args]))
    res = subprocess.run(cmd, cwd=HERE, capture_output=True)
    if check and res.returncode != 0:
        raise RuntimeError("Échec de la commande HDFS :\n" + res.stderr.decode(errors="replace"))
    return res


def md5(data: bytes) -> str:
    return hashlib.md5(data).hexdigest()

## Paramètres

In [ ]:
DATE_LIVRAISON = date.today().isoformat()   # ou une date fixe, au format "AAAA-MM-JJ"
BASE = str(ROOT / "03_data" / "warehouse" / "smartport_complet.db")

day = date.fromisoformat(DATE_LIVRAISON).isoformat()   # erreur si la date est invalide
source_md5 = md5((RAW_DIR / EVENTS).read_bytes())
folder = f"{HDFS_BASE}/ingest_date={day}"
target = f"{folder}/{EVENTS}"
print(target)

## 1. Créer le dossier daté et déposer le fichier (`mkdir`, `put`)

Une archive qui existe déjà n'est jamais écrasée.

In [ ]:
hdfs("dfs", "-mkdir", "-p", folder)
if hdfs("dfs", "-test", "-e", target, check=False).returncode == 0:
    print("Archive déjà présente : elle n'est pas écrasée, son contenu sera vérifié.")
else:
    hdfs("dfs", "-put", f"/raw/{EVENTS}", target)

## 2. Vérifier l'archive (`ls`, `head`)

In [ ]:
print(hdfs("dfs", "-ls", "-h", folder).stdout.decode(errors="replace"))
print(hdfs("dfs", "-head", target).stdout.decode(errors="replace"))

## 3. Relire depuis HDFS et comparer les empreintes MD5

Si la copie est différente, tout s'arrête et la base n'est pas modifiée.

In [ ]:
data = hdfs("dfs", "-cat", target).stdout
read_md5 = md5(data)
print(f"MD5 source : {source_md5}\nMD5 HDFS   : {read_md5}")
if read_md5 != source_md5:
    raise RuntimeError("ARRÊT : la copie relue depuis HDFS diffère de la source. Le Data Warehouse n'est pas modifié.")

## 4. Préparer la zone d'atterrissage

In [ ]:
landing = ROOT / "03_data" / "landing_hdfs" / f"ingest_date={day}"
landing.mkdir(parents=True, exist_ok=True)
(landing / EVENTS).write_bytes(data)
for name in OTHERS:
    shutil.copy(RAW_DIR / name, landing / name)

## 5. Charger le Data Warehouse depuis la copie relue

In [ ]:
print("Chargement du Data Warehouse depuis la copie relue de HDFS...")
etl = ROOT / "04_scripts" / "etl" / "etl_pipeline.py"
subprocess.run([sys.executable, str(etl), str(landing), BASE], check=True)

## 6. Enregistrer la preuve d'exécution

In [ ]:
manifest = {
    "executed_at": datetime.now().isoformat(timespec="seconds"),
    "hdfs_path": target,
    "md5_source": source_md5,
    "md5_read_from_hdfs": read_md5,
    "landing_file": (landing / EVENTS).relative_to(ROOT).as_posix(),
    "status": "OK",
}
out = ROOT / "06_results" / "ingestion_hdfs_manifest.json"
out.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Terminé. Preuve enregistrée dans {out.relative_to(ROOT)}")